# 04 - Feature Engineering
Calendar features, comfort indices (heat index, wind chill, humidity index), lag/rolling statistics, and interaction features.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
%matplotlib inline


In [2]:
import pandas as pd
from src import config
from src.feature_engineering import (add_calendar_features, add_comfort_indices,
                                      add_lag_rolling_features, add_interaction_features,
                                      build_features, select_features)

df = pd.read_csv(config.CLEAN_DATA_FILE, parse_dates=[config.DATETIME_COL])
df.shape

(12000, 41)

## Calendar features

In [3]:
df = add_calendar_features(df)
df[['year','month','day','week_of_year','quarter','is_weekend','season']].head()

2026-07-03 05:37:18 | src.feature_engineering | INFO | Added calendar features


2026-07-03 05:37:18 | src.feature_engineering | INFO | add_calendar_features finished in 0.14s


,year,month,day,week_of_year,quarter,is_weekend,season
0,2024,6,1,22,2,1,Summer
1,2024,6,2,22,2,1,Summer
2,2024,6,3,23,2,0,Summer
3,2024,6,4,23,2,0,Summer
4,2024,6,5,23,2,0,Summer


## Comfort indices

In [4]:
df = add_comfort_indices(df)
df[['temperature_celsius','feels_like_celsius','temp_feels_diff','heat_index','wind_chill','humidity_index']].head()

2026-07-03 05:37:18 | src.feature_engineering | INFO | Added comfort-index features (heat index, wind chill, humidity index)


2026-07-03 05:37:18 | src.feature_engineering | INFO | add_comfort_indices finished in 0.01s


,temperature_celsius,feels_like_celsius,temp_feels_diff,heat_index,wind_chill,humidity_index
0,1.4,0.8,-0.6,1.4,-3.361947,57.002457
1,0.2,-0.3,-0.5,0.2,-4.113379,64.299139
2,0.8,0.5,-0.3,0.8,-2.409656,69.730898
3,-0.7,-0.8,-0.1,-0.7,-0.700000,77.082716
4,0.2,0.2,0.0,0.2,0.200000,64.197531


## Lag & rolling features (per-location time series)

In [5]:
df = add_lag_rolling_features(df)
[c for c in df.columns if 'lag_' in c or 'rollmean' in c or 'rollstd' in c]

2026-07-03 05:37:18 | src.feature_engineering | INFO | Added 10 lag/rolling features


2026-07-03 05:37:18 | src.feature_engineering | INFO | add_lag_rolling_features finished in 0.16s


['temperature_celsius_lag_1',
 'temperature_celsius_lag_2',
 'temperature_celsius_lag_3',
 'temperature_celsius_lag_7',
 'temperature_celsius_rollmean_3',
 'temperature_celsius_rollstd_3',
 'temperature_celsius_rollmean_7',
 'temperature_celsius_rollstd_7',
 'temperature_celsius_rollmean_14',
 'temperature_celsius_rollstd_14']

## Interaction & polynomial features

In [6]:
df = add_interaction_features(df)
df[['humidity_temp_interaction','wind_pressure_interaction','temperature_celsius_sq']].head()

2026-07-03 05:37:18 | src.feature_engineering | INFO | Added interaction and polynomial features


2026-07-03 05:37:18 | src.feature_engineering | INFO | add_interaction_features finished in 0.01s


,humidity_temp_interaction,wind_pressure_interaction,temperature_celsius_sq
0,1000.5,20661.34,210.25
1,1425.6,16363.62,466.56
2,1060.8,14470.17,432.64
3,1230.8,10343.82,327.61
4,1004.7,10220.00,388.09


## Feature selection (top correlated with target)

In [7]:
top_features = select_features(df, top_k=15)
top_features

2026-07-03 05:37:19 | src.feature_engineering | INFO | Selected top 15 features by |correlation| with temperature_celsius


2026-07-03 05:37:19 | src.feature_engineering | INFO | select_features finished in 0.03s


['wind_chill',
 'temperature_fahrenheit',
 'feels_like_celsius',
 'feels_like_fahrenheit',
 'heat_index',
 'humidity_temp_interaction',
 'temperature_celsius_rollmean_14',
 'temperature_celsius_rollmean_7',
 'temperature_celsius_rollmean_3',
 'temperature_celsius_lag_1',
 'temperature_celsius_lag_2',
 'temperature_celsius_lag_3',
 'temperature_celsius_lag_7',
 'temperature_celsius_sq',
 'uv_index']

## Save feature-engineered dataset

In [8]:
df.to_csv(config.FEATURED_DATA_FILE, index=False)
print('Saved:', config.FEATURED_DATA_FILE, '| shape:', df.shape)

Saved: /home/claude/Weather-Trend-Forecasting/data/processed/weather_features.csv | shape: (12000, 67)
